# Disaster XAI-Net — Localization post-processing tuning (no retraining)

The trained U-Net finds building **pixels** well (test pixel F1 0.85) but often fails to separate **touching buildings** (test building recall 0.51; earthquake building F1 0.26 with ~330 buildings per tile). The pixel → building step used fixed thresholds chosen before training. This notebook improves that step **without retraining and without looking at the test set**:

1. **Test-time augmentation (TTA):** average predictions over the 8 rotations/flips of each tile.
2. **Tune the instance-splitting parameters on the VALIDATION tiles** (the same 10% held out during training): mask threshold, border threshold, seed erosion, minimum seed size and minimum building size. Objective: validation building-level F1 (IoU ≥ 0.5).
3. Evaluate the chosen setting on the **held-out test set once**, reported **next to the original pre-specified setting**.
4. Re-run the end-to-end damage evaluation, regenerate the GeoJSON and update the demo bundle.

**Kaggle inputs** (GPU on):
* `lubdham/disaster-separate-data` (notebook output, same as before)
* the **saved output of the Localization notebook** (needs `localization_results/checkpoints/best.pth` and `reproducibility/tile_split.csv`)
* `disaster-xai-results-archive` (optional, for the end-to-end damage section)

In [ ]:
import os, re, gc, json, time, glob, shutil, zipfile, datetime, itertools, importlib.util
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw
from scipy import ndimage as ndi
import multiprocessing as mp
from sklearn.metrics import f1_score, accuracy_score, cohen_kappa_score, matthews_corrcoef
import matplotlib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from skimage.segmentation import watershed
from skimage.measure import find_contours, approximate_polygon
Image.MAX_IMAGE_PIXELS = None

_env = os.environ.get
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = torch.cuda.is_available()
N_PROC = max(1, min(int(_env("TUNE_PROCS", os.cpu_count() or 2)), 8))
IOU_MATCH = 0.5
OUT = _env("TUNE_OUTPUT_ROOT", "/kaggle/working/localization_tuned")
D = {k: os.path.join(OUT, v) for k, v in {'metrics': 'metrics', 'plots': 'plots', 'pred': 'predictions',
     'bundle': 'localization_bundle', 'geo': 'geojson', 'cache': '_cache'}.items()}
for p in D.values(): os.makedirs(p, exist_ok=True)
SEARCH = [p for p in [_env("TUNE_INPUT_ROOT"), "/kaggle/input", "/kaggle/working"] if p and os.path.isdir(p)]
print("device:", device, "| worker processes:", N_PROC)

def jdump(o, p):
    t = p + ".tmp"
    with open(t, "w") as fh: json.dump(o, fh, indent=2, default=lambda x: x.item() if hasattr(x, 'item') else str(x))
    os.replace(t, p)

def find_one(pattern, prefer=None):
    hits = sorted({h for r in SEARCH for h in glob.glob(os.path.join(r, "**", pattern), recursive=True)
                   if not h.startswith(OUT)})
    if prefer: hits = sorted(hits, key=lambda h: prefer not in h)
    return hits[0] if hits else None

ORIG_CKPT = find_one("localization_results/checkpoints/best.pth") or find_one("unet_localization.pth")
SPLIT_CSV = find_one("localization_results/reproducibility/tile_split.csv") or find_one("tile_split.csv")
ORIG_TEST = find_one("localization_results/metrics/test_localization_metrics.json")
ORIG_E2E = find_one("localization_results/metrics/test_end_to_end_damage.json")
print("checkpoint:", ORIG_CKPT); print("tile split:", SPLIT_CSV)
assert ORIG_CKPT and SPLIT_CSV, "attach the SAVED output of the Localization notebook (it must contain checkpoints/best.pth)"

## 1. Data (exactly the tiles and roles used in training)

In [ ]:
hazard_base_path = _env("TUNE_DATA_ROOT")
if not hazard_base_path:
    for r in SEARCH:
        for dp, _, _ in os.walk(r):
            if os.path.basename(dp) == "hazard_data": hazard_base_path = dp; break
        if hazard_base_path: break
assert hazard_base_path, "hazard_data not found - attach lubdham/disaster-separate-data"

def prefix_of(fn):
    m = re.match(r'^(.*?)_(pre|post)_disaster\.png$', fn)
    return m.group(1) if m else None

rec = []
for hazard in sorted(os.listdir(hazard_base_path)):
    for split in ['train', 'test']:
        img_dir, lbl_dir = (os.path.join(hazard_base_path, hazard, split, d) for d in ('images', 'labels'))
        if not os.path.isdir(img_dir): continue
        files = set(os.listdir(img_dir))
        for p in sorted({prefix_of(f) for f in files if prefix_of(f)}):
            if f"{p}_pre_disaster.png" in files and f"{p}_post_disaster.png" in files:
                pre_lbl = os.path.join(lbl_dir, f"{p}_pre_disaster.json")
                rec.append({'tile': p, 'hazard': hazard,
                            'pre_path': os.path.join(img_dir, f"{p}_pre_disaster.png"),
                            'post_path': os.path.join(img_dir, f"{p}_post_disaster.png"),
                            'pre_label': pre_lbl if os.path.exists(pre_lbl) else os.path.join(lbl_dir, f"{p}_post_disaster.json"),
                            'post_label': os.path.join(lbl_dir, f"{p}_post_disaster.json")})
tiles = pd.DataFrame(rec).set_index('tile')
roles = pd.read_csv(SPLIT_CSV)
missing = set(roles['tile']) - set(tiles.index)
assert not missing, f"{len(missing)} tiles from the training split are missing in the data - wrong data version?"
val_t = tiles.loc[roles.loc[roles.role == 'val', 'tile']].reset_index()
test_t = tiles.loc[roles.loc[roles.role == 'test', 'tile']].reset_index()
assert not (set(val_t['tile']) & set(test_t['tile']))
print(f"validation tiles {len(val_t)} | test tiles {len(test_t)}")
print(val_t['hazard'].value_counts().to_string())

## 2. Labels, matching and the parameterised instance step

In [ ]:
WKT_RE = re.compile(r'\(\((.*?)\)\)')
def parse_wkt(w):
    m = WKT_RE.search(w or '')
    if not m: return None
    try: pts = [tuple(float(v) for v in c.strip().split()[:2]) for c in m.group(1).split(',')]
    except ValueError: return None
    return pts if len(pts) >= 3 else None

def read_buildings(path):
    js = json.load(open(path)); feats = js.get('features', {})
    ll = {f.get('properties', {}).get('uid'): parse_wkt(f.get('wkt')) for f in feats.get('lng_lat', []) or []}
    out = []
    for f in feats.get('xy', []) or []:
        pts = parse_wkt(f.get('wkt'))
        if pts:
            pr = f.get('properties', {})
            out.append({'uid': pr.get('uid'), 'xy': pts, 'lnglat': ll.get(pr.get('uid')), 'subtype': pr.get('subtype', 'un-classified')})
    return out

def gt_instances(buildings, size):
    im = Image.new('I', size, 0); d = ImageDraw.Draw(im)
    for k, b in enumerate(buildings, 1): d.polygon(b['xy'], fill=k)
    return np.asarray(im, np.int32)

DEFAULT = dict(thr_mask=0.5, thr_border=0.5, erode=0, min_seed=4, min_area=20)   # the original pre-specified setting

def instances(pm, pb, thr_mask, thr_border, erode, min_seed, min_area):
    fg = pm > thr_mask
    core = fg & (pb < thr_border)
    if erode: core = ndi.binary_erosion(core, iterations=erode)
    seeds, n = ndi.label(core)
    if n:
        sizes = np.bincount(seeds.ravel(), minlength=n + 1)
        seeds[(sizes < min_seed)[seeds]] = 0
    inst = watershed(-pm, seeds, mask=fg) if seeds.any() else ndi.label(fg)[0]
    sizes = np.bincount(inst.ravel())
    keep = sizes >= min_area; keep[0] = False
    remap = np.zeros(len(sizes), np.int32); remap[keep] = np.arange(1, keep.sum() + 1)
    return remap[inst]

def match(gt, pr):
    ng, npd = int(gt.max()), int(pr.max())
    if ng == 0 or npd == 0: return []
    both = (gt > 0) & (pr > 0)
    inter = np.bincount(gt[both].astype(np.int64) * (npd + 1) + pr[both], minlength=(ng + 1) * (npd + 1)).reshape(ng + 1, npd + 1)
    ag, ap = np.bincount(gt.ravel(), minlength=ng + 1), np.bincount(pr.ravel(), minlength=npd + 1)
    g, p = np.nonzero(inter); iou = inter[g, p] / (ag[g] + ap[p] - inter[g, p])
    used_g, used_p, res = set(), set(), []
    for o in np.argsort(-iou):
        if iou[o] < IOU_MATCH: break
        if g[o] in used_g or p[o] in used_p: continue
        used_g.add(g[o]); used_p.add(p[o]); res.append((int(g[o]), int(p[o]), float(iou[o])))
    return res

def polygons(inst):
    out = []
    for k, sl in enumerate(ndi.find_objects(inst), 1):
        if sl is None: continue
        ys, xs = sl; sub = inst[sl] == k
        cs = find_contours(np.pad(sub, 1).astype(float), 0.5); poly = None
        if cs:
            c = approximate_polygon(max(cs, key=len), tolerance=1.0)
            poly = [[float(x - 1 + xs.start), float(y - 1 + ys.start)] for y, x in c]
        if not poly or len(poly) < 3:
            poly = [[xs.start, ys.start], [xs.stop, ys.start], [xs.stop, ys.stop], [xs.start, ys.stop], [xs.start, ys.start]]
        out.append({'id': k, 'bbox': [float(xs.start), float(ys.start), float(xs.stop), float(ys.stop)],
                    'area': int(sub.sum()), 'polygon': poly})
    return out
print("ready")

## 3. Load the trained U-Net (architecture from the saved bundle code) and cache probabilities

Probabilities for every validation and test tile are computed once (with and without TTA) and stored as 8-bit arrays, so the parameter search is pure post-processing.

In [ ]:
LOC_PY = find_one("localization_results/localization_bundle/localization_inference.py") or find_one("localization_inference.py")
assert LOC_PY, "localization_bundle/localization_inference.py not found in the attached Localization output"
spec = importlib.util.spec_from_file_location("loc_inf_orig", LOC_PY)
LI = importlib.util.module_from_spec(spec); spec.loader.exec_module(LI)
ck = torch.load(ORIG_CKPT, map_location='cpu', weights_only=False)
META = ck['meta']
model = LI.UNet(META['encoder']); model.load_state_dict(ck['state_dict']); model.to(device).eval()
MEAN, STD = np.array(META['mean'], np.float32), np.array(META['std'], np.float32)
print("model:", {k: META[k] for k in ('encoder', 'epoch', 'val_pixel_f1')})

D4 = [(k, f) for k in range(4) for f in (False, True)]
@torch.no_grad()
def probs(img_u8, tta):
    x = torch.from_numpy(((img_u8.astype(np.float32) / 255 - MEAN) / STD).transpose(2, 0, 1).copy())[None].to(device)
    H, W = x.shape[-2:]; ph, pw = (32 - H % 32) % 32, (32 - W % 32) % 32
    if ph or pw: x = F.pad(x, (0, pw, 0, ph), mode='reflect')
    acc = 0
    for k, f in (D4 if tta else [(0, False)]):
        xi = torch.rot90(x, k, (2, 3)); xi = torch.flip(xi, (3,)) if f else xi
        with torch.amp.autocast('cuda', enabled=AMP):
            y = torch.sigmoid(model(xi).float())
        y = torch.flip(y, (3,)) if f else y
        acc = acc + torch.rot90(y, -k, (2, 3))
    p = (acc / (len(D4) if tta else 1))[0, :, :H, :W].cpu().numpy()
    return p

def cache_split(df, tag):
    path = os.path.join(D['cache'], f"{tag}.npz")
    if os.path.exists(path):
        z = np.load(path, allow_pickle=True); return {k: z[k] for k in z.files}
    P0, P8, GT = [], [], []
    for r in tqdm(df.itertuples(), total=len(df), desc=f"predict {tag}"):
        img = np.asarray(Image.open(r.pre_path).convert('RGB'))
        P0.append((probs(img, False) * 255).round().astype(np.uint8))
        P8.append((probs(img, True) * 255).round().astype(np.uint8))
        g_ = gt_instances(read_buildings(r.pre_label), (img.shape[1], img.shape[0]))
        GT.append(g_.astype(np.uint16) if g_.max() < 65535 else g_)
    out = {}
    for k, v in (('p_plain', P0), ('p_tta', P8), ('gt', GT)):
        a = np.empty(len(v), object); a[:] = v; out[k] = a
    np.savez(path, **out)
    return out

VAL = cache_split(val_t, 'val')
print("validation probabilities cached")

## 4. Parameter search on the VALIDATION tiles

Objective: building-level F1 over all validation buildings. Ties are broken by fewer parameter changes from the original setting. Each configuration is evaluated in parallel across tiles.

In [ ]:
_ARR = {}
def _eval_tile(args):
    i, mode, cfg = args
    p = _ARR[mode][i]; pm, pb = p[0] / 255.0, p[1] / 255.0
    gt = _ARR['gt'][i]
    inst = instances(pm, pb, **cfg)
    m = match(gt, inst)
    return len(m), int(gt.max()), int(inst.max())

def evaluate(arr, cfg, mode):
    global _ARR
    _ARR = arr
    n = len(arr['gt'])
    with mp.get_context('fork').Pool(N_PROC) as pool:
        res = pool.map(_eval_tile, [(i, mode, cfg) for i in range(n)], chunksize=max(1, n // (N_PROC * 4)))
    tp, ngt, npd = map(sum, zip(*res))
    pr, rc = tp / max(npd, 1), tp / max(ngt, 1)
    return {'building_precision': pr, 'building_recall': rc, 'building_f1': 2 * pr * rc / max(pr + rc, 1e-9)}

t0 = time.time()
rows = []
for mode in ['p_plain', 'p_tta']:
    rows.append({'mode': mode, **DEFAULT, **evaluate(VAL, DEFAULT, mode)})
base = pd.DataFrame(rows)
print(base.round(4).to_string(index=False), f"\n({time.time()-t0:.0f}s for 2 configurations)")
MODE = 'p_tta' if base.loc[1, 'building_f1'] >= base.loc[0, 'building_f1'] else 'p_plain'
print("TTA", "helps -> used" if MODE == 'p_tta' else "does not help -> not used")

GRID = dict(thr_mask=[0.4, 0.5], thr_border=[0.2, 0.3, 0.4, 0.5], erode=[0, 1, 2], min_seed=[4, 16], min_area=[20, 40])
GRID = {k: [v for v in vs] for k, vs in GRID.items()}
if _env("TUNE_SMALL_GRID") == "1":
    GRID = dict(thr_mask=[0.5], thr_border=[0.3, 0.5], erode=[0, 1], min_seed=[4], min_area=[20])
configs = [dict(zip(GRID, v)) for v in itertools.product(*GRID.values())]
print(f"searching {len(configs)} configurations on {len(VAL['gt'])} validation tiles ...")
rows = []
for cfg in tqdm(configs, desc="grid"):
    rows.append({**cfg, **evaluate(VAL, cfg, MODE)})
grid = pd.DataFrame(rows)
grid['n_changed'] = sum((grid[k] != v).astype(int) for k, v in DEFAULT.items())
grid = grid.sort_values(['building_f1', 'n_changed'], ascending=[False, True]).reset_index(drop=True)
grid.to_csv(os.path.join(D['metrics'], "val_grid_search.csv"), index=False)
BEST = {k: (int(grid.loc[0, k]) if k in ('erode', 'min_seed', 'min_area') else float(grid.loc[0, k])) for k in DEFAULT}
print(grid.head(10).round(4).to_string(index=False))
print("\nselected on validation:", BEST, "| TTA:", MODE == 'p_tta')
jdump({'selected': BEST, 'tta': MODE == 'p_tta', 'objective': 'validation building F1 (IoU>=0.5)',
       'n_validation_tiles': len(VAL['gt']), 'grid': GRID, 'default': DEFAULT,
       'baseline_val': base.to_dict('records'), 'best_val': grid.iloc[0].to_dict()},
      os.path.join(D['metrics'], "tuning_selection.json"))

# sensitivity: best F1 for each value of each parameter (shows the choice is not a lucky spike)
fig, axes = plt.subplots(1, len(GRID), figsize=(3.2 * len(GRID), 3), sharey=True)
for ax, k in zip(axes, GRID):
    s = grid.groupby(k)['building_f1'].max()
    ax.plot(s.index.astype(str), s.values, marker='o'); ax.set_title(k); ax.grid(alpha=.3)
axes[0].set_ylabel('best val building F1')
plt.tight_layout(); plt.savefig(os.path.join(D['plots'], "val_sensitivity.png"), dpi=150); plt.show(); plt.close(fig)

## 5. FINAL HELD-OUT TEST — original setting vs. validation-tuned setting (once)

In [ ]:
TEST = cache_split(test_t, 'test')
del model; gc.collect(); torch.cuda.empty_cache()

def test_eval(cfg, mode, keep_instances=False):
    tp_px = fp_px = fn_px = 0; per = []; kept = {}
    for i, r in enumerate(tqdm(test_t.itertuples(), total=len(test_t), desc="test", leave=False)):
        p = TEST[mode][i]; pm, pb = p[0] / 255.0, p[1] / 255.0; gt = TEST['gt'][i]
        pmask, t = pm > cfg['thr_mask'], gt > 0
        tp_px += int((pmask & t).sum()); fp_px += int((pmask & ~t).sum()); fn_px += int((~pmask & t).sum())
        inst = instances(pm, pb, **cfg); m = match(gt, inst)
        per.append({'tile': r.tile, 'hazard': r.hazard, 'n_gt': int(gt.max()), 'n_pred': int(inst.max()), 'matched': len(m)})
        if keep_instances: kept[r.tile] = {'buildings': polygons(inst), 'matches': m, 'size': [gt.shape[1], gt.shape[0]]}
    per = pd.DataFrame(per)
    def bf(g):
        pr, rc = g['matched'].sum() / max(g['n_pred'].sum(), 1), g['matched'].sum() / max(g['n_gt'].sum(), 1)
        return pr, rc, 2 * pr * rc / max(pr + rc, 1e-9)
    pr, rc, f = bf(per)
    overall = {'pixel_f1': 2 * tp_px / max(2 * tp_px + fp_px + fn_px, 1), 'building_precision': pr,
               'building_recall': rc, 'building_f1': f, 'n_pred_buildings': int(per['n_pred'].sum()),
               'n_gt_buildings': int(per['n_gt'].sum())}
    byh = pd.DataFrame([{'hazard': h, **dict(zip(['building_precision', 'building_recall', 'building_f1'], bf(g)))}
                        for h, g in per.groupby('hazard')])
    return overall, byh, per, kept

orig_o, orig_h, _, _ = test_eval(DEFAULT, 'p_plain')
tuned_o, tuned_h, tuned_per, PRED = test_eval(BEST, MODE, keep_instances=True)
cmp = pd.DataFrame([{'setting': 'original (pre-specified)', **orig_o}, {'setting': 'tuned on validation' + (' + TTA' if MODE == 'p_tta' else ''), **tuned_o}])
cmp.to_csv(os.path.join(D['metrics'], "test_localization_original_vs_tuned.csv"), index=False)
hz = orig_h.merge(tuned_h, on='hazard', suffixes=('_original', '_tuned'))
hz.to_csv(os.path.join(D['metrics'], "test_building_f1_by_hazard_original_vs_tuned.csv"), index=False)
tuned_per.to_csv(os.path.join(D['metrics'], "test_per_tile_tuned.csv"), index=False)
jdump({'selected': BEST, 'tta': MODE == 'p_tta', **tuned_o}, os.path.join(D['metrics'], "test_localization_metrics_tuned.json"))
jdump(PRED, os.path.join(D['pred'], "test_instances_tuned.json"))
print("=" * 70); print("HELD-OUT TEST: original vs tuned post-processing"); print("=" * 70)
print(cmp.round(4).to_string(index=False)); print(); print(hz.round(4).to_string(index=False))
if ORIG_TEST:
    o = json.load(open(ORIG_TEST))
    print(f"\nconsistency check - original setting reproduces the training notebook: building F1 {orig_o['building_f1']:.4f} vs {o['building_f1']:.4f}")

## 6. End-to-end damage assessment with the tuned buildings

Same procedure and conventions as the Localization notebook (missed buildings count as predicted `no-damage`).

In [ ]:
DAMAGE = ['no-damage', 'minor-damage', 'major-damage', 'destroyed']
bundle_py = None
hits = sorted({h for r in SEARCH for h in glob.glob(os.path.join(r, "**", "deployment_bundle", "inference.py"), recursive=True)})
fusion = [h for h in hits if 'disaster_xai_net_master_results' in h or 'disaster-xai-results-archive' in h]
bundle_py = (fusion or hits or [None])[0]
E2E = None
if not bundle_py:
    print("damage deployment_bundle not attached - skipped")
else:
    print("USING damage bundle:", bundle_py)
    spec = importlib.util.spec_from_file_location("dxn_damage", bundle_py)
    dmg = importlib.util.module_from_spec(spec); spec.loader.exec_module(dmg)
    dmg.load_models(os.path.dirname(bundle_py))
    rows, sev_rows = [], []
    for r in tqdm(test_t.itertuples(), total=len(test_t), desc="end-to-end damage"):
        pre_b = read_buildings(r.pre_label)
        if not pre_b: continue
        sub_by_uid = {b['uid']: b['subtype'] for b in read_buildings(r.post_label)}
        P = PRED[r.tile]
        gt_req = [{'building_id': f"g{k}", 'wkt': 'POLYGON ((' + ', '.join(f"{x} {y}" for x, y in b['xy']) + '))'} for k, b in enumerate(pre_b, 1)]
        oracle = {x['building_id']: x for x in dmg.predict_full(r.pre_path, r.post_path, buildings=gt_req)['buildings']}
        req = [{'building_id': f"p{b['id']}", 'bbox': b['bbox']} for b in P['buildings']]
        predr = {x['building_id']: x for x in dmg.predict_full(r.pre_path, r.post_path, buildings=req)['buildings']} if req else {}
        for b in P['buildings']:
            x = predr[f"p{b['id']}"]
            sev_rows.append({'tile': r.tile, 'pred_building_id': b['id'], 'severity': x['severity'], 'p_damaged': x['p_damaged'], 'confidence': x['confidence']})
        g2p = {g: p for g, p, _ in P['matches']}
        for k, b in enumerate(pre_b, 1):
            sub = sub_by_uid.get(b['uid'], b['subtype'])
            if sub not in DAMAGE: continue
            pm_ = g2p.get(k)
            rows.append({'tile': r.tile, 'hazard': r.hazard, 'gt': sub, 'oracle': oracle[f"g{k}"]['severity'],
                         'matched': pm_ is not None, 'pred': predr[f"p{pm_}"]['severity'] if pm_ is not None else None})
    e = pd.DataFrame(rows); e.to_csv(os.path.join(D['pred'], "end_to_end_buildings_tuned.csv"), index=False)
    pd.DataFrame(sev_rows).to_csv(os.path.join(D['pred'], "end_to_end_predicted_building_severity_tuned.csv"), index=False)
    idx = {n: i for i, n in enumerate(DAMAGE)}
    def sm(y, p):
        y, p = [idx[v] for v in y], [idx[v] for v in p]
        return {'n': len(y), 'macro_f1': f1_score(y, p, average='macro', labels=range(4), zero_division=0),
                'accuracy': accuracy_score(y, p), 'qwk': cohen_kappa_score(y, p, weights='quadratic'), 'mcc': matthews_corrcoef(y, p)}
    mt = e[e['matched']]
    E2E = {'oracle_all_gt_buildings': sm(e['gt'], e['oracle']), 'oracle_on_matched': sm(mt['gt'], mt['oracle']),
           'predicted_footprints_on_matched': sm(mt['gt'], mt['pred']),
           'end_to_end_missed_as_no_damage': sm(e['gt'], e['pred'].fillna('no-damage')),
           'detection_recall_by_true_class_and_hazard': e.pivot_table(index='gt', columns='hazard', values='matched', aggfunc='mean').round(4).to_dict()}
    jdump(E2E, os.path.join(D['metrics'], "test_end_to_end_damage_tuned.json"))
    tab = pd.DataFrame({k: v for k, v in E2E.items() if 'macro_f1' in v}).T
    if ORIG_E2E:
        o = json.load(open(ORIG_E2E))
        tab['macro_f1_original'] = [o.get(k, {}).get('macro_f1') for k in tab.index]
    tab.round(4).to_csv(os.path.join(D['metrics'], "test_end_to_end_damage_tuned_table.csv"))
    print(tab.round(4).to_string())
    dmg._STATE.clear(); del dmg; gc.collect(); torch.cuda.empty_cache()

## 7. Figures, GeoJSON and the updated demo bundle

In [ ]:
# figure: densest earthquake tile + a typical cyclone tile, ground truth vs original vs tuned
pick = []
for hz_ in ['earthquake', 'cyclone']:
    g = tuned_per[tuned_per['hazard'] == hz_]
    if len(g): pick.append(g.sort_values('n_gt').iloc[-1 if hz_ == 'earthquake' else len(g) // 2]['tile'])
if pick:
    fig, axes = plt.subplots(len(pick), 3, figsize=(15, 5 * len(pick)))
    axes = np.atleast_2d(axes)
    for rr, t in enumerate(pick):
        i = test_t.index[test_t['tile'] == t][0]
        img = np.asarray(Image.open(test_t.loc[i, 'pre_path']).convert('RGB'))
        gt = TEST['gt'][i]
        p0 = TEST['p_plain'][i] / 255.0; p1 = TEST[MODE][i] / 255.0
        i0, i1 = instances(p0[0], p0[1], **DEFAULT), instances(p1[0], p1[1], **BEST)
        m0, m1 = len(match(gt, i0)), len(match(gt, i1))
        for c, (lab, inst, txt) in enumerate([('ground truth', gt, f"{gt.max()} buildings"),
                                             ('original', i0, f"{i0.max()} predicted, {m0} matched"),
                                             ('tuned', i1, f"{i1.max()} predicted, {m1} matched")]):
            ax = axes[rr, c]; ax.imshow(img)
            ax.imshow(np.ma.masked_equal(inst % 20 + (inst > 0), 0), cmap='tab20', alpha=.55, interpolation='nearest')
            ax.set_title(f"{test_t.loc[i, 'hazard']} - {lab}: {txt}", fontsize=10); ax.axis('off')
    plt.tight_layout(); plt.savefig(os.path.join(D['plots'], "test_examples_original_vs_tuned.png"), dpi=130, bbox_inches='tight')
    plt.show(); plt.close(fig)

# GeoJSON (affine pixel -> lon/lat fitted on each tile's label vertices)
SEV = {}
p_ = os.path.join(D['pred'], "end_to_end_predicted_building_severity_tuned.csv")
if os.path.exists(p_):
    for x in pd.read_csv(p_).itertuples():
        SEV[(x.tile, int(x.pred_building_id))] = {'severity': x.severity, 'p_damaged': float(x.p_damaged), 'severity_confidence': float(x.confidence)}
features = []
for r in test_t.itertuples():
    src, dst = [], []
    for b in read_buildings(r.post_label):
        if b['lnglat'] and len(b['lnglat']) == len(b['xy']): src += b['xy']; dst += b['lnglat']
    if len(src) < 3: continue
    A = np.c_[np.asarray(src), np.ones(len(src))]
    if np.linalg.matrix_rank(A) < 3: continue
    coef = np.linalg.lstsq(A, np.asarray(dst), rcond=None)[0]
    for b in PRED[r.tile]['buildings']:
        ring = (np.c_[np.asarray(b['polygon']), np.ones(len(b['polygon']))] @ coef).tolist()
        if ring[0] != ring[-1]: ring.append(ring[0])
        features.append({'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [ring]},
                         'properties': {'tile': r.tile, 'hazard': r.hazard, 'pred_building_id': b['id'], 'area_px': b['area'],
                                        'source': 'U-Net (tuned post-processing)', **SEV.get((r.tile, b['id']), {})}})
jdump({'type': 'FeatureCollection', 'features': features}, os.path.join(D['geo'], "predicted_buildings_test_tuned.geojson"))
print("GeoJSON features:", len(features))

# updated bundle: same weights, tuned post-processing (+ TTA flag)
B = D['bundle']
shutil.copy2(ORIG_CKPT, os.path.join(B, "unet_localization.pth"))
src_code = open(LOC_PY).read()
ADDON = '''

# ---- tuned post-processing (selected on validation tiles; see postprocess_config.json) ----
_PP = None
def _pp():
    global _PP
    if _PP is None:
        _PP = json.load(open(os.path.join(os.path.dirname(os.path.abspath(__file__)), 'postprocess_config.json')))
    return _PP

@torch.no_grad()
def predict_masks(pre):
    S = load(); a = _img(pre); m = S['meta']
    t = torch.from_numpy(((a / 255.0 - np.array(m['mean'])) / np.array(m['std'])).transpose(2, 0, 1)).float()[None].to(S['dev'])
    H, W = t.shape[-2:]; ph, pw = (32 - H % 32) % 32, (32 - W % 32) % 32
    if ph or pw: t = F.pad(t, (0, pw, 0, ph), mode='reflect')
    views = [(k, f) for k in range(4) for f in (False, True)] if _pp()['tta'] else [(0, False)]
    acc = 0
    for k, f in views:
        x = torch.rot90(t, k, (2, 3)); x = torch.flip(x, (3,)) if f else x
        y = torch.sigmoid(S['model'](x).float()); y = torch.flip(y, (3,)) if f else y
        acc = acc + torch.rot90(y, -k, (2, 3))
    p = (acc / len(views))[0, :, :H, :W].cpu().numpy()
    return p[0], p[1]

def predict_buildings(pre):
    c = _pp(); pm, pb = predict_masks(pre)
    fg = pm > c['thr_mask']; core = fg & (pb < c['thr_border'])
    if c['erode']: core = ndi.binary_erosion(core, iterations=c['erode'])
    seeds, n = ndi.label(core)
    if n:
        sizes = np.bincount(seeds.ravel(), minlength=n + 1); seeds[(sizes < c['min_seed'])[seeds]] = 0
    inst = watershed(-pm, seeds, mask=fg) if (_SK and seeds.any()) else ndi.label(fg)[0]
    out = []
    for k, sl in enumerate(ndi.find_objects(inst), 1):
        if sl is None: continue
        sub = inst[sl] == k
        if sub.sum() < c['min_area']: continue
        ys, xs = sl; poly = None
        if _SK:
            cs = find_contours(np.pad(sub, 1).astype(float), 0.5)
            if cs:
                cc = approximate_polygon(max(cs, key=len), tolerance=1.0)
                poly = [[float(x - 1 + xs.start), float(y - 1 + ys.start)] for y, x in cc]
        if not poly or len(poly) < 3:
            poly = [[xs.start, ys.start], [xs.stop, ys.start], [xs.stop, ys.stop], [xs.start, ys.stop], [xs.start, ys.start]]
        out.append({'id': len(out) + 1, 'bbox': [float(xs.start), float(ys.start), float(xs.stop), float(ys.stop)],
                    'area': int(sub.sum()), 'polygon': poly})
    return out
'''
with open(os.path.join(B, "localization_inference.py"), "w") as fh:
    fh.write(src_code.replace("import os, io", "import os, io, json", 1) + ADDON)
jdump({**BEST, 'tta': MODE == 'p_tta', 'selected_on': 'validation building F1'}, os.path.join(B, "postprocess_config.json"))
jdump({'test_tuned': tuned_o, 'test_original': orig_o, 'end_to_end_tuned': E2E}, os.path.join(B, "model_card.json"))

spec = importlib.util.spec_from_file_location("loc_inf_tuned", os.path.join(B, "localization_inference.py"))
LT = importlib.util.module_from_spec(spec); spec.loader.exec_module(LT)
LT.load(os.path.join(B, "unet_localization.pth"))
t0_ = test_t.iloc[0]
nb_, bd_ = len(PRED[t0_['tile']]['buildings']), len(LT.predict_buildings(t0_['pre_path']))
print(f"bundle check on {t0_['tile']}: notebook {nb_} buildings, bundle {bd_}")
assert abs(nb_ - bd_) <= max(3, 0.05 * nb_), "updated bundle disagrees with the notebook"

## 8. Small ZIP for the paper (no weights) + full ZIP

In [ ]:
shutil.rmtree(D['cache'], ignore_errors=True)
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
small = f"/kaggle/working/localization_tuned_paper_{stamp}.zip" if os.path.isdir("/kaggle/working") else f"{OUT}_paper.zip"
with zipfile.ZipFile(small, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, fs in os.walk(OUT):
        for fn in fs:
            if fn.endswith(('.pth', '.tmp')): continue
            full = os.path.join(root, fn); zf.write(full, os.path.join("localization_tuned", os.path.relpath(full, OUT)))
print(f"paper ZIP: {small} ({os.path.getsize(small)/1e6:.1f} MB)")
print(cmp.round(4).to_string(index=False))
try:
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(small, "/kaggle/working") if small.startswith("/kaggle/working") else small))
except Exception:
    pass